# 03 – Datenpipeline

> Hinweis: Dieses Notebook ist ein exploratives Arbeitsartefakt. Der aktuelle, benotungsrelevante Stand steht in `../reports/FINAL_REPORT.pdf`; alte Zelloutputs wurden entfernt und die Zellen sollen bei Bedarf neu ausgefuehrt werden.


**Projekt:** Polymarket Reddit Sentiment  
**Kurs:** Data Wrangling & Engineering (FHNW)

Dieses Notebook dokumentiert die vollständige Datenpipeline vom Rohdaten-Abruf bis zum analysierbaren Datensatz.

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   INGEST     │───▶│   CLEAN      │───▶│  TRANSFORM   │───▶│   OUTPUT     │
│              │    │              │    │              │    │              │
│ Reddit API   │    │ Missing vals │    │ Sentiment    │    │ CSV / DF     │
│ Polymarket   │    │ Duplikate    │    │ Aggregation  │    │ Dashboard    │
│              │    │ Outlier      │    │ Join         │    │              │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

---

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import os
import pandas as pd
import numpy as np
from scipy import stats
from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

from src import reddit, polymarket, sentiment

print('Libraries geladen.')

## 1. Pipeline-Konfiguration

In [ ]:
@dataclass
class PipelineConfig:
    """Konfiguration der Datenpipeline."""
    query: str = 'Bitcoin'
    subreddits: list = field(default_factory=lambda: ['investing', 'stocks', 'worldnews', 'CryptoCurrency'])
    post_limit: int = 100
    sentiment_model: str = sentiment.MODEL_ROBERTA
    winsorize_lower: float = 0.05
    winsorize_upper: float = 0.95
    min_title_length: int = 5
    output_dir: str = '../data'

config = PipelineConfig(query='Bitcoin')
print(config)

## 2. Pipeline-Klasse

In [ ]:
class SentimentPipeline:
    """
    Vollständige Datenpipeline:
      1. Ingest  – Daten von Reddit + Polymarket abrufen
      2. Clean   – Fehlende Werte, Duplikate, Ausreisser behandeln
      3. Transform – Sentiment berechnen, Daten zusammenführen
      4. Output  – Ergebnisse speichern
    """

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.log = []

    def _log(self, step: str, msg: str):
        entry = f'[{step}] {msg}'
        self.log.append(entry)
        print(entry)

    # ── Schritt 1: Ingest ────────────────────────────────────────────────────
    def ingest(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        self._log('INGEST', f'Lade Reddit-Posts für "{self.config.query}"...')
        raw_posts = reddit.get_posts(
            self.config.query,
            self.config.subreddits,
            self.config.post_limit
        )
        self._log('INGEST', f'{len(raw_posts)} Posts geladen')

        self._log('INGEST', 'Lade Polymarket-Daten...')
        try:
            markets = polymarket.get_markets(limit=50)
            self._log('INGEST', f'{len(markets)} Märkte geladen')
        except Exception as e:
            self._log('INGEST', f'Polymarket nicht erreichbar: {e} – verwende leeren DataFrame')
            markets = pd.DataFrame(columns=['id', 'question', 'probability', 'category', 'volume'])

        return raw_posts, markets

    # ── Schritt 2: Clean ────────────────────────────────────────────────────
    def clean(self, posts: pd.DataFrame, markets: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        posts = posts.copy()
        markets = markets.copy()

        # Fehlende Werte
        posts['text'] = posts['text'].fillna('')
        self._log('CLEAN', f'text: fehlende Werte mit "" gefüllt')

        if 'probability' in markets.columns and markets['probability'].isnull().any():
            markets['probability_missing'] = markets['probability'].isnull().astype(int)
            markets['probability'] = markets['probability'].fillna(markets['probability'].median())
            self._log('CLEAN', 'probability: Median-Imputation + Indikatorvariable')

        if 'category' in markets.columns:
            markets['category'] = markets['category'].fillna('Unknown')

        # Duplikate
        n_before = len(posts)
        posts = posts.drop_duplicates(subset='id', keep='first')
        posts = posts.drop_duplicates(subset='title', keep='first')
        self._log('CLEAN', f'Duplikate entfernt: {n_before - len(posts)}')

        # Ausreisser (Winsorisierung)
        for col in ['score', 'num_comments']:
            if col in posts.columns:
                lo = posts[col].quantile(self.config.winsorize_lower)
                hi = posts[col].quantile(self.config.winsorize_upper)
                posts[col] = posts[col].clip(lower=lo, upper=hi)
                self._log('CLEAN', f'{col}: Winsorisiert [{lo:.0f}, {hi:.0f}]')

        # Textreinigung
        def clean_text(t):
            if not isinstance(t, str):
                return ''
            t = re.sub(r'http\S+', '', t)
            t = re.sub(r'\[deleted\]|\[removed\]', '', t)
            return re.sub(r'\s+', ' ', t).strip()

        posts['title_clean'] = posts['title'].apply(clean_text)
        posts['text_clean']  = posts['text'].apply(clean_text)

        # Leere Posts markieren
        posts['is_empty_content'] = (
            (posts['title_clean'].str.len() < self.config.min_title_length) &
            (posts['text_clean'].str.len() < self.config.min_title_length)
        ).astype(int)
        self._log('CLEAN', f'Leere Posts markiert: {posts["is_empty_content"].sum()}')

        return posts, markets

    # ── Schritt 3: Transform ────────────────────────────────────────────────
    def transform(self, posts: pd.DataFrame, markets: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        # Sentiment berechnen
        posts = sentiment.analyze(posts, model=self.config.sentiment_model)
        self._log('TRANSFORM', f'Sentiment berechnet für {len(posts)} Posts (Modell: {self.config.sentiment_model})')

        # Datum normalisieren
        if 'created_utc' in posts.columns:
            posts['date'] = pd.to_datetime(posts['created_utc']).dt.date

        # Tages-Aggregat
        if 'date' in posts.columns:
            daily = posts.groupby('date').agg(
                mean_compound=('compound', 'mean'),
                post_count=('id', 'count'),
                positive_pct=('sentiment_label', lambda x: (x == 'positive').mean())
            ).reset_index()
            self._log('TRANSFORM', f'Tages-Aggregat erstellt: {len(daily)} Tage')
        else:
            daily = pd.DataFrame()

        return posts, daily

    # ── Schritt 4: Output ───────────────────────────────────────────────────
    def save(self, posts: pd.DataFrame, daily: pd.DataFrame):
        os.makedirs(self.config.output_dir, exist_ok=True)
        posts.to_csv(f'{self.config.output_dir}/reddit_clean.csv', index=False)
        if not daily.empty:
            daily.to_csv(f'{self.config.output_dir}/daily_sentiment.csv', index=False)
        self._log('OUTPUT', f'Gespeichert in {self.config.output_dir}/')

    # ── Vollständige Pipeline ───────────────────────────────────────────────
    def run(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        print('=' * 50)
        print(f'Pipeline Start: "{self.config.query}"')
        print('=' * 50)

        raw_posts, markets = self.ingest()
        clean_posts, clean_markets = self.clean(raw_posts, markets)
        final_posts, daily = self.transform(clean_posts, clean_markets)
        self.save(final_posts, daily)

        print('=' * 50)
        print(f'Pipeline abgeschlossen. {len(final_posts)} Posts verarbeitet.')
        print('=' * 50)
        return final_posts, daily

print('Pipeline-Klasse definiert.')

## 2b. Datenharmonisierung: Reddit ↔ Polymarket

Die Pipeline verbindet zwei ergänzende Datenquellen:

| Schritt | Reddit | Polymarket |
|---|---|---|
| Quelle | Public JSON API () | CLOB + Gamma API () |
| Einheit | Post (Titel + Text + Upvotes) | Markt (Frage + Wahrscheinlichkeit) |
| Verbindungslogik | Keyword-Suche | Marktfrage als Anker |

**Join-Strategie (run_analysis.py / run_bulk.py):**
1. Polymarket-Marktfrage →  extrahiert 4 relevante Substantive
2. Keywords → Reddit-Suche über 7 Subreddits → n Posts pro Markt
3. Posts → Twitter-RoBERTa →  als aggregierter Sentiment-Score
4.  ↔  → F1 Korrelationsanalyse (Notebook 04)

**Harmonisierung gleicher Inhalte (VADER vs. RoBERTa):**  
Dieselben Reddit-Posts wurden zusätzlich mit VADER und Twitter-RoBERTa verglichen (Notebook 04).  
Dies erfüllt die Vorgabe „gleicher Inhalt, Fokus Harmonisierung“:
- **VADER**: regelbasiert, schnell, kein Download (Ø Skala: −0.2 bis +0.2)
- **RoBERTa**: transformer-basiert, social-media-optimiert (Ø Skala: −0.4 bis +0.4)
- Z-Score-Normalisierung in Notebook 04 macht beide Skalen vergleichbar

## 3. Pipeline ausführen

In [ ]:
pipeline = SentimentPipeline(config)
posts_final, daily_df = pipeline.run()

## 4. Pipeline-Log anzeigen

In [ ]:
print('\nPipeline-Log:')
for entry in pipeline.log:
    print(f'  {entry}')

## 5. Ergebnis-Überblick

In [ ]:
print(f'Output-Shape: {posts_final.shape}')
print(f'Spalten:      {list(posts_final.columns)}')
print()
posts_final.head(3)

In [ ]:
if not daily_df.empty:
    print('Tages-Aggregat (daily_sentiment):')
    print(daily_df.to_string(index=False))

## 6. Pipeline – Verschiedene Themen vergleichen

In [ ]:
topics  = ['Bitcoin', 'Trump', 'Climate Change']
results = {}

for topic in topics:
    cfg  = PipelineConfig(query=topic, post_limit=50)
    pipe = SentimentPipeline(cfg)
    try:
        df, _ = pipe.run()
        agg   = sentiment.aggregate(df)
        agg['n_posts'] = len(df)           # Anzahl Posts für Annotation
        results[topic] = agg
        print(f'{topic}: mean={agg["mean_compound"]:+.3f}  label={agg["label"]}  n={agg["n_posts"]}')
    except Exception as e:
        print(f'{topic}: Fehler – {e}')


In [ ]:
import matplotlib.pyplot as plt
import datetime

if results:
    topics_list = list(results.keys())
    scores      = [results[t]['mean_compound'] for t in topics_list]
    n_posts     = [results[t].get('n_posts', 0) for t in topics_list]
    bar_colors  = ['#2ecc71' if s >= 0.05 else '#e74c3c' if s <= -0.05 else '#f1c40f'
                   for s in scores]

    # Sentiment-Counts fuer Panel 2
    pos_counts = [results[t]['counts'].get('positive', 0) for t in topics_list]
    neu_counts = [results[t]['counts'].get('neutral',  0) for t in topics_list]
    neg_counts = [results[t]['counts'].get('negative', 0) for t in topics_list]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Panel 1: Mean Compound
    bars = ax1.bar(topics_list, scores, color=bar_colors, edgecolor='white', width=0.5)
    ax1.axhline(0,      color='black', linewidth=0.8)
    ax1.axhline( 0.05,  color='green', ls='--', alpha=0.6, lw=1, label='positiv (0.05)')
    ax1.axhline(-0.05,  color='red',   ls='--', alpha=0.6, lw=1, label='negativ (-0.05)')
    ax1.set_ylabel('Oe Compound Score (RoBERTa)')
    ax1.set_title('Themenvergleich: Oe Sentiment-Score')
    ax1.legend(fontsize=8)
    ax1.grid(axis='y', alpha=0.2)

    for bar, score, n in zip(bars, scores, n_posts):
        y_offset = 0.007 if score >= 0 else -0.007
        va = 'bottom' if score >= 0 else 'top'
        ax1.text(bar.get_x() + bar.get_width()/2, score + y_offset,
                 f'{score:+.3f} (n={n})', ha='center', va=va, fontsize=9, fontweight='bold')

    # Panel 2: Sentiment-Verteilung (gestapelt)
    x = range(len(topics_list))
    ax2.bar(x, pos_counts, label='Positiv', color='#2ecc71', edgecolor='white', alpha=0.85)
    ax2.bar(x, neu_counts, bottom=pos_counts, label='Neutral',
            color='#f1c40f', edgecolor='white', alpha=0.85)
    ax2.bar(x, neg_counts, bottom=[p+n for p,n in zip(pos_counts, neu_counts)],
            label='Negativ', color='#e74c3c', edgecolor='white', alpha=0.85)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(topics_list)
    ax2.set_ylabel('Anzahl Posts')
    ax2.set_title('Sentiment-Verteilung pro Thema')
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.2)

    model_name = 'Twitter-RoBERTa'
    today = datetime.date.today().isoformat()
    plt.suptitle('Pipeline-Ergebnis: ' + model_name + ' | ' + today,
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../notebooks/pipeline_themenvergleich.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Gespeichert: notebooks/pipeline_themenvergleich.png')
